# 04 · Deep Learning พื้นฐาน (ช่วงที่ 4)

> **เป้าหมาย:** เข้าใจว่า neural network ทำงานอย่างไรตั้งแต่ neuron เดียวจนถึงการฝึกทั้งโครงข่าย และ **อ่าน loss curve แล้วบอกได้ว่า overfit หรือไม่**

**เวลาที่ใช้:** ~2 ชม. | ต้องใช้: numpy, matplotlib, scikit-learn (ส่วน PyTorch ตอนท้ายเป็นทางเลือก)

### ทำไมโน้ตนี้เขียนด้วย NumPy ก่อน PyTorch?
เฟรมเวิร์กอย่าง PyTorch ซ่อนรายละเอียด (`loss.backward()` บรรทัดเดียวทำ backprop ทั้งหมด) ถ้าเขียนเองด้วย NumPy ครั้งหนึ่ง เธอจะเห็นว่าข้างในคืออะไร แล้วพอใช้ PyTorch จะรู้ว่าแต่ละบรรทัด "แทน" อะไร **ตอนท้ายมีโค้ด PyTorch เทียบกับของที่เขียนเอง**

| ส่วน | หัวข้อ | ตรงกับโน้ตของเธอ |
|---|---|---|
| 1 | Neuron + Activation | neural network concepts |
| 2 | Layers, shape, forward pass | neural network concepts |
| 3 | Loss (MSE, cross-entropy) | loss curve |
| 4 | Gradient + Backpropagation (+ ตรวจด้วยตัวเลข) | neural network concepts |
| 5 | ฝึกโครงข่ายจริง ๆ + loss curve | **loss curve** |
| 6 | Optimizer: SGD / Momentum / **Adam** | **adams** |
| 7 | **Learning rate / Batch size / Epoch** ทดลองเอง | learning rate, batch size, epoch |
| 8 | **Overfit**: ทำให้เกิดจริง + early stopping + weight decay | **overfit** |
| 9 | PyTorch เทียบกับที่เขียนเอง + ภาพรวม TensorFlow/sklearn | tensorflow pytorch sklearn |
| 10 | สรุป + resources + repos |

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Circle
import time, warnings; warnings.filterwarnings("ignore")
from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split
plt.rcParams["figure.dpi"] = 110
rng = np.random.default_rng(0)

## 1. Neuron + Activation

Neuron หนึ่งตัวทำแค่สองอย่าง:
```
 x1 ──w1──┐
 x2 ──w2──┼─→  z = w1·x1 + w2·x2 + b  ─→  a = activation(z)  ─→ output
 x3 ──w3──┘
```
1. **รวมแบบถ่วงน้ำหนัก** `z = w·x + b` (w = น้ำหนัก = "สำคัญแค่ไหน", b = bias = "เกณฑ์ตั้งต้น")
2. ส่งผ่าน **activation function** เพื่อ "ดัดเส้น"

**ทำไมต้องมี activation?** ถ้าไม่มี การต่อ neuron กี่ชั้นก็ยังเป็นเส้นตรงเท่าเดิม (เส้นตรงซ้อนเส้นตรงก็คือเส้นตรง) activation ที่ไม่เป็นเส้นตรงคือสิ่งที่ทำให้โครงข่ายเรียนรู้รูปทรงโค้งซับซ้อนได้

In [ ]:
def sigmoid(z): return 1 / (1 + np.exp(-z))
def relu(z):    return np.maximum(0, z)

z = np.linspace(-6, 6, 300)
fig, axes = plt.subplots(1, 3, figsize=(13, 3.2))
for ax, (name, f) in zip(axes, [("sigmoid: squashes to (0,1)", sigmoid), ("tanh: squashes to (-1,1)", np.tanh), ("ReLU: max(0, z)", relu)]):
    ax.plot(z, f(z), lw=2); ax.axhline(0, c="gray", lw=0.5); ax.axvline(0, c="gray", lw=0.5); ax.set_title(name, fontsize=10); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

# neuron เดียวด้วยมือ: เปลี่ยน w, b แล้วดูผล
x = np.array([1.0, 2.0, -1.0])
for w, b in [(np.array([0.5, 0.5, 0.5]), 0.0), (np.array([0.5, 0.5, 0.5]), -2.0), (np.array([-1.0, 0.2, 1.0]), 0.0)]:
    zz = w @ x + b; print(f"w={w}, b={b:>4}: z={zz:+.2f}  sigmoid(z)={sigmoid(zz):.3f}  relu(z)={relu(zz):.2f}")

**สังเกต:** เปลี่ยน `b` จาก 0 เป็น -2 ทำให้ z ลดลง (bias = เลื่อนเกณฑ์) เปลี่ยนเครื่องหมายของ `w` ก็เปลี่ยนว่า input ตัวไหน "ดึงขึ้น/ดึงลง"

**ReLU** เป็นตัวเลือกมาตรฐานของ hidden layer ในปัจจุบัน (ง่ายและฝึกเร็ว) | **sigmoid** อยู่ท้ายสุดในงาน 2 คลาส | **softmax** (ดูส่วนที่ 3) อยู่ท้ายสุดในงานหลายคลาส

> 🧪 **Mini challenge 1.1:** ลองหา `w` และ `b` ของ neuron ที่มี input เดียว ให้ `sigmoid` ออกมา ≈ 0.5 เมื่อ x = 3 (คำใบ้: ต้องให้ z = 0)
>
> 🧪 **Mini challenge 1.2:** ทำไม sigmoid/tanh ที่ z = ±6 แทบแบนราบ? ถ้ากราฟแบน "ความชัน" เป็นอย่างไร และจะส่งผลอย่างไรตอนเรียนรู้ (เจอคำตอบใน ส่วนที่ 4)

## 2. Layers, shape, forward pass

เอา neuron หลายตัวเรียงเป็น **layer** แล้วต่อหลาย layer = **neural network** (MLP)

```
 Input        Hidden 1       Hidden 2       Output
(2 features)  (4 neurons)    (4 neurons)    (2 classes)
   x1 ───●────●────●────●
   x2 ───●────●────●────●────●  → softmax → ความน่าจะเป็นของแต่ละคลาส
         ●────●────●────●────●
```
ทุกเส้นเชื่อม = หนึ่งน้ำหนัก (weight) สิ่งที่ "โมเดลเรียนรู้" ก็คือชุดน้ำหนักและ bias เหล่านี้ทั้งหมด

**ท่า "เมทริกซ์":** layer ทั้งชั้นคำนวณพร้อมกันด้วย  `Z = X @ W + b`
- `X` shape `(N, d_in)`: N ตัวอย่าง, d_in feature
- `W` shape `(d_in, d_out)`, `b` shape `(d_out,)`
- `Z` shape `(N, d_out)`

**ฝึกอ่าน shape ให้คล่อง** — error ส่วนใหญ่ใน deep learning คือ shape ไม่ตรง

In [ ]:
def draw_mlp(sizes, ax, names):
    xs = np.linspace(0, 1, len(sizes)); pos = []
    for x, n in zip(xs, sizes):
        ys = np.linspace(0.25, 0.95, n) if n > 1 else [0.6]
        ys = ys if n > 2 else np.linspace(0.45, 0.75, n); pos.append([(x, y) for y in ys])
    for l in range(len(sizes) - 1):
        for p in pos[l]:
            for q in pos[l + 1]: ax.plot([p[0], q[0]], [p[1], q[1]], c="#cbd5e1", lw=0.8, zorder=1)
    cols = ["#bfdbfe"] + ["#fde68a"] * (len(sizes) - 2) + ["#bbf7d0"]
    for layer, col in zip(pos, cols):
        for p in layer: ax.scatter(*p, s=420, c=col, edgecolor="k", zorder=2)
    for x, lab in zip(xs, names): ax.text(x, 0.05, lab, ha="center", va="center", fontsize=9)
    ax.set_xlim(-0.12, 1.12); ax.set_ylim(-0.05, 1.05); ax.axis("off")
fig, ax = plt.subplots(figsize=(7, 3.4)); draw_mlp([2, 4, 4, 2], ax, ["input (2)", "hidden 1 (4)", "hidden 2 (4)", "output (2)"]); ax.set_title("A 2-4-4-2 network"); plt.show()

# forward pass ทีละบรรทัด พร้อมดู shape
N = 5
X = rng.normal(size=(N, 2))
W1, b1 = rng.normal(size=(2, 4)), np.zeros(4)
W2, b2 = rng.normal(size=(4, 4)), np.zeros(4)
W3, b3 = rng.normal(size=(4, 2)), np.zeros(2)

Z1 = X @ W1 + b1;  A1 = relu(Z1);   print("X", X.shape, "-> Z1", Z1.shape, "-> A1", A1.shape)
Z2 = A1 @ W2 + b2; A2 = relu(Z2);   print("A1", A1.shape, "-> Z2", Z2.shape, "-> A2", A2.shape)
Z3 = A2 @ W3 + b3;                  print("A2", A2.shape, "-> Z3 (logits)", Z3.shape)

n_params = sum(a.size for a in [W1, b1, W2, b2, W3, b3]); print("จำนวนพารามิเตอร์ทั้งหมด:", n_params, "= (2*4+4) + (4*4+4) + (4*2+2)")

> 🧪 **Mini challenge 2.1:** ถ้าโครงข่ายเป็น `[63, 128, 64, 20]` (63 feature = พิกัดมือหนึ่งข้าง, 20 คำ) มีกี่พารามิเตอร์? เขียนสูตรแล้วคำนวณ (คำใบ้: ต่อ layer คือ `d_in*d_out + d_out`)
>
> ❓ **คิดดู:** พารามิเตอร์เยอะเกินจำนวนข้อมูลฝึก เสี่ยงอะไร? (กลับไปดูภาพ overfit ใน notebook 02)

## 3. Loss: วัดว่าทำนายผิดเท่าไร

โมเดลที่ยังไม่ได้ฝึกให้ผลมั่ว เราต้องมีตัวเลขเดียวที่บอกว่า "ผิดมากแค่ไหน" = **loss** (ยิ่งต่ำยิ่งดี) การฝึกคือการลด loss

- **Regression** (ทำนายตัวเลข): MSE = เฉลี่ยของ (ทำนาย − จริง)²
- **Classification** (ทำนายคลาส): **cross-entropy**

**Softmax:** เปลี่ยนคะแนนดิบ (logits) ของแต่ละคลาส ให้เป็นความน่าจะเป็นที่รวมกันได้ 1
```
softmax(z)_k = exp(z_k) / Σ_j exp(z_j)
```
**Cross-entropy ของตัวอย่างเดียว:**  `loss = −log( p_ของคลาสที่ถูกต้อง )`  
ถ้าโมเดลให้ p ของคำตอบจริงสูง → loss ต่ำ; ถ้าให้ต่ำ → loss พุ่ง (โดยเฉพาะถ้า "มั่นใจแต่ผิด")

In [ ]:
def softmax(z):
    z = z - z.max(axis=1, keepdims=True)           # ลบค่าสูงสุดกัน exp ล้น (ผลลัพธ์เท่าเดิม)
    e = np.exp(z); return e / e.sum(axis=1, keepdims=True)

logits = np.array([[2.0, 1.0, 0.1]])
p = softmax(logits); print("logits:", logits[0], "-> probabilities:", p[0].round(3), "| sum =", p.sum().round(3))
for true_class in range(3): print(f"  ถ้าคำตอบจริงคือคลาส {true_class}: loss = -log({p[0, true_class]:.3f}) = {-np.log(p[0, true_class]):.3f}")

pp = np.linspace(0.01, 1, 200)
fig, ax = plt.subplots(figsize=(5.5, 3.3)); ax.plot(pp, -np.log(pp), lw=2)
ax.set_xlabel("probability given to the CORRECT class"); ax.set_ylabel("cross-entropy loss"); ax.set_title("Confident-and-wrong is punished hard"); ax.grid(alpha=0.3); plt.show()

**สังเกต:** p → 1 แล้ว loss → 0; p → 0 แล้ว loss พุ่งขึ้นเร็วมาก โมเดลที่ "มั่นใจแต่ผิด" โดนลงโทษหนักที่สุด

> 🧪 **Mini challenge 3.1:** เปลี่ยน `logits` เป็น `[10.0, 1.0, 0.1]` p ของคลาส 0 เป็นเท่าไร? ถ้าคำตอบจริงคือคลาส 2 loss เป็นเท่าไร (ตัวเลขใหญ่ไหม ทำไม)
>
> 🧪 **Mini challenge 3.2:** ถ้า 20 คลาสและโมเดล "มั่ว" ให้ทุกคลาส p = 1/20 loss ต่อตัวอย่างเป็นเท่าไร? (`-log(1/20)`) — นี่คือ *ค่าอ้างอิงของ loss ตอนเริ่มฝึก* (ของจริงจะไม่เป๊ะเพราะน้ำหนักสุ่ม เช่นใน notebook นี้เริ่มที่ ~0.9 สำหรับ 2 คลาสที่อ้างอิง 0.69) ถ้า loss เริ่มต้นห่างจากค่านี้หลายเท่า ให้สงสัยบั๊กหรือการ init

## 4. Gradient + Backpropagation

เราอยากปรับน้ำหนักให้ loss ลดลง ต้องรู้ว่า **ขยับน้ำหนักแต่ละตัวขึ้นนิดเดียว loss จะเปลี่ยนเท่าไร** = *gradient* (อนุพันธ์ ∂loss/∂w)

```
w ← w − lr · ∂loss/∂w       # ก้าวสวนทางความชัน (gradient descent)
```
โครงข่ายมีน้ำหนักเป็นหมื่นล้านตัว ใช้ **backpropagation** = กฎลูกโซ่ (chain rule) คำนวณ gradient ทุกตัวอย่างมีประสิทธิภาพ โดยไล่ย้อนจาก loss ไปทาง input

### ตัวอย่างเล็กที่สุด: คำนวณมือได้
```
x → [ z = w·x + b ] → [ a = σ(z) ] → [ L = (a − y)² ]
```
กฎลูกโซ่: `∂L/∂w = ∂L/∂a · ∂a/∂z · ∂z/∂w = 2(a−y) · σ'(z) · x`  โดย `σ'(z) = σ(z)(1−σ(z))`

In [ ]:
x_, y_ = 1.5, 1.0
w_, b_ = 0.4, -0.2

def forward_tiny(w, b):
    z = w * x_ + b; a = sigmoid(z); return z, a, (a - y_) ** 2

z, a, L = forward_tiny(w_, b_)
dL_da = 2 * (a - y_); da_dz = a * (1 - a); dz_dw = x_; dz_db = 1.0
grad_w, grad_b = dL_da * da_dz * dz_dw, dL_da * da_dz * dz_db        # chain rule

# ตรวจด้วยวิธี "ขยับนิดเดียวแล้วดูว่า loss เปลี่ยนเท่าไร" (numerical gradient)
eps = 1e-6
num_w = (forward_tiny(w_ + eps, b_)[2] - forward_tiny(w_ - eps, b_)[2]) / (2 * eps)
num_b = (forward_tiny(w_, b_ + eps)[2] - forward_tiny(w_, b_ - eps)[2]) / (2 * eps)
print(f"z={z:.3f}, a={a:.3f}, loss={L:.4f}")
print(f"dL/dw: backprop={grad_w:.8f}  numerical={num_w:.8f}")
print(f"dL/db: backprop={grad_b:.8f}  numerical={num_b:.8f}")
print("\nσ'(z) =", round(da_dz, 4), "  <- ถ้า z ใหญ่มาก σ'(z) ≈ 0 -> gradient หายไป (vanishing gradient) ตอบ challenge 1.2")

**สังเกต:** gradient จากกฎลูกโซ่ตรงกับ numerical gradient (ต่างกันแค่ทศนิยมหลังๆ) → เราเขียนอนุพันธ์ถูก วิธี "gradient check" นี้ใช้ตรวจ backprop ที่เขียนเองได้เสมอ

**ภาพรวม backprop:** forward เก็บค่าระหว่างทาง (z, a ของทุก layer) → backward ไล่จากหลังมาหน้า แต่ละ layer รับ "ความผิดที่ส่งมาจากข้างหลัง" แล้วคูณกับอนุพันธ์ของตัวเอง นั่นคือทั้งหมด

ต่อไปเราสร้างโครงข่ายเต็มรูปแบบ (หลาย layer, ReLU, softmax + cross-entropy) ด้วย NumPy สูตรสำคัญที่ใช้: เมื่อใช้ softmax + cross-entropy จะได้ `∂L/∂logits = (p − y_onehot) / N` ซึ่งเรียบง่ายมาก

In [ ]:
# ---------- MLP เขียนเอง: parameters เก็บเป็น list [W0, b0, W1, b1, ...] ----------
def init_params(sizes, rng):
    theta = []
    for d_in, d_out in zip(sizes[:-1], sizes[1:]):
        theta += [rng.normal(0, np.sqrt(2.0 / d_in), (d_in, d_out)), np.zeros(d_out)]    # He init เหมาะกับ ReLU
    return theta

def forward(theta, X):
    A, Z = [X], []
    L = len(theta) // 2
    for l in range(L):
        z = A[-1] @ theta[2 * l] + theta[2 * l + 1]; Z.append(z)
        A.append(relu(z) if l < L - 1 else softmax(z))              # hidden: ReLU, ท้ายสุด: softmax
    return A, Z

def cross_entropy(P, y): return -np.mean(np.log(P[np.arange(len(y)), y] + 1e-12))

def backward(theta, A, Z, y, l2=0.0):
    L = len(theta) // 2; n = len(y); grads = [None] * len(theta)
    dZ = A[-1].copy(); dZ[np.arange(n), y] -= 1; dZ /= n               # (p - onehot) / N
    for l in reversed(range(L)):
        grads[2 * l] = A[l].T @ dZ + l2 * theta[2 * l]                 # dW (+ L2 / weight decay)
        grads[2 * l + 1] = dZ.sum(axis=0)                              # db
        if l > 0: dZ = (dZ @ theta[2 * l].T) * (Z[l - 1] > 0)         # ส่งความผิดย้อนไป layer ก่อนหน้า (× อนุพันธ์ ReLU)
    return grads

def accuracy(theta, X, y): return (forward(theta, X)[0][-1].argmax(1) == y).mean()

# ---------- Gradient check: เทียบ backprop กับ numerical gradient ----------
rs = np.random.default_rng(1)
theta_chk = [t.astype(np.float64) for t in init_params([2, 5, 4, 3], rs)]
Xc, yc = rs.normal(size=(6, 2)), rs.integers(0, 3, 6)
A, Z = forward(theta_chk, Xc); g = backward(theta_chk, A, Z, yc)
max_rel = 0
for k in range(len(theta_chk)):
    for idx in np.ndindex(*theta_chk[k].shape):
        old = theta_chk[k][idx]; e = 1e-6
        theta_chk[k][idx] = old + e; lp = cross_entropy(forward(theta_chk, Xc)[0][-1], yc)
        theta_chk[k][idx] = old - e; lm = cross_entropy(forward(theta_chk, Xc)[0][-1], yc)
        theta_chk[k][idx] = old; num = (lp - lm) / (2 * e)
        max_rel = max(max_rel, abs(num - g[k][idx]) / max(1e-8, abs(num) + abs(g[k][idx])))
print("ความต่างสัมพัทธ์สูงสุดระหว่าง backprop กับ numerical gradient:", f"{max_rel:.2e}", "(ควรเล็กมาก เช่นต่ำกว่า 1e-5)")

**อ่านผล:** ถ้าตัวเลขเล็กมาก แปลว่า backprop ที่เราเขียนถูกต้องกับ *ทุกพารามิเตอร์* ของโครงข่ายนี้ (นี่คือสิ่งที่ `loss.backward()` ของ PyTorch ทำให้เรา)

> 🧪 **Mini challenge 4.1:** ลองแกล้งทำให้ backprop ผิด เช่นลบบรรทัด `* (Z[l - 1] > 0)` แล้วรัน gradient check ใหม่ ตัวเลขเป็นอย่างไร?
>
> ❓ **คิดดู:** ถ้า z ของ ReLU เป็นลบทั้งหมด gradient ที่ส่งย้อนผ่าน neuron นั้นเป็นเท่าไร? (นี่คือที่มาของ "dying ReLU")

## 5. ฝึกโครงข่ายจริง ๆ + อ่าน loss curve

ลูปฝึกมาตรฐาน (จำไว้ — ทุกเฟรมเวิร์กหน้าตาเหมือนกัน):
```
สำหรับแต่ละ EPOCH (วนข้อมูลครบทั้งชุด 1 รอบ):
    สับข้อมูล, แบ่งเป็น BATCH ย่อย
    สำหรับแต่ละ BATCH:
        1. forward        → ได้การทำนาย
        2. loss           → วัดความผิด
        3. backward       → คำนวณ gradient
        4. optimizer step → ปรับน้ำหนัก
    วัด loss/accuracy บน train และ validation  → เก็บไว้วาด loss curve
```
**ศัพท์:** *epoch* = วนครบข้อมูล 1 รอบ | *batch size* = ใช้กี่ตัวอย่างต่อการปรับน้ำหนัก 1 ครั้ง | *iteration* = การปรับ 1 ครั้ง (ข้อมูล N ตัว, batch B → N/B iterations ต่อ epoch)

In [ ]:
# ---------- Optimizers (ใช้ในส่วนที่ 6) ----------
def make_opt(name, theta, lr, beta1=0.9, beta2=0.999, eps=1e-8, momentum=0.9):
    return {"name": name, "lr": lr, "b1": beta1, "b2": beta2, "eps": eps, "mom": momentum, "t": 0,
            "m": [np.zeros_like(p) for p in theta], "v": [np.zeros_like(p) for p in theta]}

def opt_step(o, theta, grads):
    o["t"] += 1
    for i, (p, g) in enumerate(zip(theta, grads)):
        if o["name"] == "sgd":
            p -= o["lr"] * g
        elif o["name"] == "momentum":
            o["m"][i] = o["mom"] * o["m"][i] + g;  p -= o["lr"] * o["m"][i]
        elif o["name"] == "adam":
            o["m"][i] = o["b1"] * o["m"][i] + (1 - o["b1"]) * g              # ค่าเฉลี่ยของ gradient (ทิศทางที่คงที่)
            o["v"][i] = o["b2"] * o["v"][i] + (1 - o["b2"]) * g * g          # ค่าเฉลี่ยของ gradient^2 (ขนาดความผันผวน)
            m_hat = o["m"][i] / (1 - o["b1"] ** o["t"]); v_hat = o["v"][i] / (1 - o["b2"] ** o["t"])   # แก้ bias ช่วงต้น
            p -= o["lr"] * m_hat / (np.sqrt(v_hat) + o["eps"])

# ---------- ลูปฝึก ----------
def train(Xtr, ytr, Xva, yva, sizes, lr=0.01, epochs=100, bs=32, opt="adam", l2=0.0, seed=0, snapshots=()):
    r = np.random.default_rng(seed); theta = init_params(sizes, r); o = make_opt(opt, theta, lr)
    hist = {"tr_loss": [], "va_loss": [], "tr_acc": [], "va_acc": [], "snap": {}}
    n = len(ytr)
    for ep in range(epochs + 1):
        if ep > 0:
            perm = r.permutation(n)
            for i in range(0, n, bs):
                idx = perm[i:i + bs]; A, Z = forward(theta, Xtr[idx]); opt_step(o, theta, backward(theta, A, Z, ytr[idx], l2))
        hist["tr_loss"].append(cross_entropy(forward(theta, Xtr)[0][-1], ytr)); hist["va_loss"].append(cross_entropy(forward(theta, Xva)[0][-1], yva))
        hist["tr_acc"].append(accuracy(theta, Xtr, ytr)); hist["va_acc"].append(accuracy(theta, Xva, yva))
        if ep in snapshots: hist["snap"][ep] = [p.copy() for p in theta]
    hist["theta"] = theta; return hist

def plot_boundary(ax, theta, X, y, title=""):
    xx, yy = np.meshgrid(np.linspace(-2.2, 3.2, 200), np.linspace(-1.6, 2.2, 200))
    zz = forward(theta, np.c_[xx.ravel(), yy.ravel()])[0][-1].argmax(1).reshape(xx.shape)
    ax.contourf(xx, yy, zz, alpha=0.25, cmap="coolwarm"); ax.scatter(X[:, 0], X[:, 1], c=y, cmap="coolwarm", edgecolor="k", s=16); ax.set_title(title, fontsize=9)

# ---------- ข้อมูล: make_moons (2 คลาสรูปพระจันทร์เสี้ยว) ----------
Xm, ym = make_moons(n_samples=600, noise=0.25, random_state=0)
Xtr, Xva, ytr, yva = train_test_split(Xm, ym, test_size=0.4, random_state=0, stratify=ym)
print("train:", Xtr.shape, "| val:", Xva.shape)

h = train(Xtr, ytr, Xva, yva, sizes=[2, 16, 16, 2], lr=0.01, epochs=200, bs=32, opt="adam", snapshots=(0, 5, 20, 200))
print(f"loss เริ่มต้น = {h['tr_loss'][0]:.3f}  (ค่า 'ทายมั่ว' แบบเท่าๆ กันคือ -log(1/2) = {-np.log(0.5):.3f}; น้ำหนักสุ่มเริ่มต้นทำให้ไม่เป๊ะ แต่ควรอยู่ในระดับเดียวกัน)")
best_h = int(np.argmin(h["va_loss"])); print(f"val loss ต่ำสุดที่ epoch {best_h} = {h['va_loss'][best_h]:.3f}  (หลังจากนั้นเป็นอย่างไร ดูกราฟถัดไป)")
print(f"จบ: train loss={h['tr_loss'][-1]:.3f}, val loss={h['va_loss'][-1]:.3f} | train acc={h['tr_acc'][-1]:.3f}, val acc={h['va_acc'][-1]:.3f}")

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(15, 3.4))
for ax, ep in zip(axes, [0, 5, 20, 200]): plot_boundary(ax, h["snap"][ep], Xtr, ytr, f"epoch {ep}")
plt.suptitle("The decision boundary is LEARNED step by step", y=1.03); plt.tight_layout(); plt.show()

fig, axes = plt.subplots(1, 2, figsize=(12, 3.8))
axes[0].plot(h["tr_loss"], label="train loss"); axes[0].plot(h["va_loss"], label="validation loss"); axes[0].set_xlabel("epoch"); axes[0].set_ylabel("cross-entropy"); axes[0].legend(); axes[0].grid(alpha=0.3); axes[0].set_title("LOSS CURVE")
axes[1].plot(h["tr_acc"], label="train acc"); axes[1].plot(h["va_acc"], label="validation acc"); axes[1].set_xlabel("epoch"); axes[1].legend(); axes[1].grid(alpha=0.3); axes[1].set_title("ACCURACY CURVE")
plt.tight_layout(); plt.show()

### ลองอ่านกราฟของเราเองก่อน
- ช่วงแรก train และ val ลดพร้อมกัน → เรียนรู้จริง
- ถัดมา **train ยังค่อยๆ ลด แต่ val loss ต่ำสุดแล้วค่อยๆ กลับขึ้น** นี่คือ *overfit ระยะเริ่มต้น* (ดู epoch ที่ print ไว้ว่าจุดต่ำสุดอยู่ที่เท่าไร)
- แต่ **val accuracy ยังนิ่งอยู่ที่ราวๆ เดิม** ข้อสังเกตสำคัญ: loss ไวต่อ "ความมั่นใจ" ของโมเดล (มั่นใจแล้วผิดโดนลงโทษหนัก) accuracy นับแค่ถูก/ผิด ดังนั้น **loss มักเตือน overfit ก่อน accuracy** นี่คือเหตุผลที่ต้องดู loss curve ไม่ใช่แค่ accuracy

### วิธีอ่าน loss curve (สำคัญที่สุดของโน้ตนี้)
กราฟนี้คือ "สุขภาพ" ของการฝึก:

| สิ่งที่เห็น | ความหมาย | ทำอะไร |
|---|---|---|
| train ลด, val ลดตาม (ใกล้กัน) | ฝึกได้ปกติ | ฝึกต่อ |
| train ลด แต่ **val เริ่มกลับขึ้น** | **overfit** | หยุดเร็ว (early stopping), regularize, เพิ่มข้อมูล, ลดขนาดโมเดล |
| ทั้งคู่ลดช้ามาก/สูงค้าง | underfit หรือ lr เล็กไป | เพิ่ม capacity, เพิ่ม lr, ฝึกนานขึ้น |
| loss กระโดด/เป็น NaN/พุ่งขึ้น | **lr ใหญ่เกิน** หรือมีบั๊ก | ลด lr, ตรวจข้อมูล |
| train loss ไม่ลดเลยจากค่าตั้งต้น | บั๊ก (ข้อมูล/label/lr) | gradient check, ลองฝึกกับข้อมูล 10 ตัวอย่างให้ overfit ให้ได้ก่อน |

> 💡 **เทคนิคจริงที่นักวิจัยใช้:** ก่อนฝึกเต็มรูป ลองฝึกกับข้อมูลแค่ ~10-20 ตัวอย่างดู ถ้าโมเดลทำ loss ลงเกือบ 0 ไม่ได้ แปลว่ามีบั๊ก

> 🧪 **Mini challenge 5.1:** เปลี่ยน `sizes=[2, 16, 16, 2]` เป็น `[2, 2, 2]` (เล็กมาก) แล้วดูขอบเขตและ loss curve (underfit?) แล้วลอง `[2, 64, 64, 2]`

## 6. Optimizer: SGD → Momentum → Adam

**Optimizer** = วิธีที่ใช้ gradient ปรับน้ำหนัก

| วิธี | แนวคิด | สูตร (ย่อ) |
|---|---|---|
| **SGD** | ก้าวตาม gradient ตรงๆ | `w ← w − lr·g` |
| **Momentum** | มี "แรงเฉื่อย" สะสมทิศทางก่อนหน้า ลดการแกว่ง | `m ← μ·m + g;  w ← w − lr·m` |
| **Adam** | Momentum + ปรับขนาดก้าว **ของแต่ละพารามิเตอร์** ตามความผันผวนของ gradient | `m ← β₁m+(1−β₁)g; v ← β₂v+(1−β₂)g²; w ← w − lr·m̂/(√v̂+ε)` |

Adam ค่าเริ่มต้นที่นิยม: `β₁=0.9, β₂=0.999, ε=1e-8` (ค่าที่เสนอในบทความต้นฉบับของ Adam) ได้รับความนิยมเพราะทนทานต่อการเลือก lr ไม่ดีนัก

ดูทั้งสามเคลื่อนที่บนพื้นผิว loss ที่ "ยาวแคบ" (ทิศหนึ่งชัน อีกทิศแบน) ซึ่งเป็นสถานการณ์ที่ SGD ทำได้ไม่ดี:

In [ ]:
def f(p):  return p[0] ** 2 / 20 + p[1] ** 2        # หุบเขายาวแคบ: แกน x แบน, แกน y ชัน
def gf(p): return np.array([p[0] / 10, 2 * p[1]])

def run_opt(name, lr, steps=60, start=(-8.0, 2.0)):
    p = np.array(start); o = make_opt(name, [p], lr); path = [p.copy()]
    for _ in range(steps):
        opt_step(o, [p], [gf(p)]); path.append(p.copy())
    return np.array(path)

paths = {"SGD (lr=0.4)": run_opt("sgd", 0.4), "Momentum (lr=0.1)": run_opt("momentum", 0.1), "Adam (lr=0.6)": run_opt("adam", 0.6)}
xx, yy = np.meshgrid(np.linspace(-9, 3, 200), np.linspace(-3, 3, 200)); zz = xx ** 2 / 20 + yy ** 2
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].contour(xx, yy, zz, levels=20, cmap="Greys", alpha=0.6)
for (name, pth), c in zip(paths.items(), ["#2563eb", "#16a34a", "#dc2626"]): axes[0].plot(pth[:, 0], pth[:, 1], ".-", c=c, lw=1, ms=4, label=name)
axes[0].scatter(0, 0, marker="*", s=200, c="gold", edgecolor="k", zorder=5); axes[0].legend(fontsize=8); axes[0].set_title("Paths on an elongated valley (★ = minimum)")
for (name, pth), c in zip(paths.items(), ["#2563eb", "#16a34a", "#dc2626"]): axes[1].plot([f(q) for q in pth], c=c, label=name)
axes[1].set_yscale("log"); axes[1].set_xlabel("step"); axes[1].set_ylabel("loss f"); axes[1].legend(fontsize=8); axes[1].grid(alpha=0.3); axes[1].set_title("Loss vs step")
plt.tight_layout(); plt.show()
for name, pth in paths.items(): print(f"{name:20s} final loss after 60 steps = {f(pth[-1]):.3e}")

**สังเกต (จากภาพที่รันได้):**
- **SGD (น้ำเงิน):** ลงแกนชัน (y) เกือบทันที แล้ว *คลานช้าๆ* ตามแกนแบน (x) เพราะ gradient ตามแกนนั้นเล็กมาก → loss ลดเรียบแต่ช้า
- **Momentum (เขียว):** สะสมแรงเฉื่อยจึงเร็วขึ้นตามแกนแบน แต่ **พุ่งเลยและแกว่ง** ไปมา (loss กราฟเป็นลูกคลื่น)
- **Adam (แดง):** ก้าวใหญ่ขึ้นในทิศที่ gradient เล็ก จึงเข้าใกล้ ★ เร็ว แต่ก็แกว่งเช่นกัน
- ตัวเลขหลัง 60 ก้าวที่ print: Adam < Momentum < SGD (loss ยิ่งต่ำยิ่งดี) **เฉพาะพื้นผิวและ lr ที่เลือกในตัวอย่างนี้** — ไม่ใช่กฎว่า Adam ชนะเสมอ ในงานจริงบางครั้ง SGD + momentum ที่จูนดีให้ผลดีกว่า Adam

> 🧪 **Mini challenge 6.1:** เปลี่ยน lr ของแต่ละวิธี (เช่น SGD เป็น 0.9 และ 1.05) เกิดอะไรขึ้น? ขอบการ diverge ของ SGD อยู่ที่ 2/ความโค้งสูงสุด (ที่นี่ความโค้งแกน y = 2)
>
> 🧪 **Mini challenge 6.2:** ลองเปลี่ยนจุดเริ่ม `start=(-8.0, 2.0)` เป็นจุดอื่น แล้วเทียบอันดับของสามวิธี

## 7. Learning rate / Batch size / Epoch — ทดลองเอง

สามตัวนี้คือ **hyperparameters** หลักที่เราตั้งเอง (ต่างจากน้ำหนักที่โมเดลเรียนรู้เอง) ใช้ validation set เป็นตัวตัดสิน

### 7.1 Learning rate

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 3.8))
for lr, c in [(0.0005, "#9ca3af"), (0.01, "#2563eb"), (0.3, "#f59e0b"), (3.0, "#dc2626")]:
    hh = train(Xtr, ytr, Xva, yva, [2, 16, 16, 2], lr=lr, epochs=60, bs=32, opt="adam")
    axes[0].plot(hh["tr_loss"], c=c, label=f"lr={lr}"); axes[1].plot(hh["va_acc"], c=c, label=f"lr={lr}")
    print(f"lr={lr:<7} final train loss={hh['tr_loss'][-1]:.3f}  val acc={hh['va_acc'][-1]:.3f}")
axes[0].set_title("train loss"); axes[1].set_title("validation accuracy")
for ax in axes: ax.set_xlabel("epoch"); ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

**สังเกต:** lr เล็กเกิน → ลดช้า | lr กลาง → ลดดีและนิ่ง | lr ใหญ่เกิน → loss แกว่งหรือไม่ลด/ผลแย่ (ดูตัวเลข print ประกอบกราฟ) นี่คือ hyperparameter ที่ **สำคัญที่สุด** ตัวหนึ่ง

### 7.2 Batch size
batch เล็ก: อัปเดตบ่อย (เสียงรบกวนสูง) แต่ละก้าวเร็ว | batch ใหญ่: gradient แม่นกว่า แต่อัปเดตน้อยครั้งต่อ epoch

In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 3.8)); n = len(ytr)
for bs, c in [(4, "#dc2626"), (32, "#2563eb"), (n, "#16a34a")]:
    t0 = time.perf_counter(); hh = train(Xtr, ytr, Xva, yva, [2, 16, 16, 2], lr=0.01, epochs=40, bs=bs, opt="adam"); dt = time.perf_counter() - t0
    ax.plot(hh["tr_loss"], c=c, label=f"batch={bs if bs < n else 'full (' + str(n) + ')'}")
    print(f"batch={bs:<4} iterations/epoch={int(np.ceil(n / bs)):<4} train loss={hh['tr_loss'][-1]:.3f}  val acc={hh['va_acc'][-1]:.3f}  เวลา={dt:.1f}s")
ax.set_xlabel("epoch"); ax.set_ylabel("train loss"); ax.legend(); ax.grid(alpha=0.3); ax.set_title("Same epochs, different batch size"); plt.show()

**สังเกต:** ที่จำนวน epoch เท่ากัน batch เล็กทำ iteration เยอะกว่ามาก จึงลด loss ต่อ epoch ได้เร็วกว่า แต่ใช้เวลาจริงนานกว่าในโค้ด Python นี้ (บน GPU จริง batch ใหญ่ขนานได้ดี จึงเป็นอีกเรื่องหนึ่ง) ค่าที่นิยม: 16-256

### 7.3 Epoch
epoch มากไป → overfit, น้อยไป → underfit วิธีที่ดีกว่า "เดาจำนวน epoch" คือ **ฝึกนานๆ แล้วดู validation เพื่อหยุดที่จุดดีที่สุด** (early stopping — ส่วนที่ 8)

> 🧪 **Mini challenge 7.1:** ลอง `lr=0.3` กับ `opt="sgd"` แทน Adam ที่ batch เดิม ผลต่างจาก Adam ไหม? (SGD ไวต่อ lr กว่ามาก)
>
> 🧪 **Mini challenge 7.2:** ทำกราฟ "ความแม่นยำเมื่อ validation เท่ากัน vs. จำนวน *iteration* ทั้งหมด" (ไม่ใช่ epoch) สำหรับ 3 batch size ผลที่เห็นเปลี่ยนไปไหม?

## 8. Overfit: ทำให้เกิดจริง แล้วแก้

สูตรสร้าง overfit: **ข้อมูลฝึกน้อย + โมเดลใหญ่ + ฝึกนาน** ใช้ข้อมูลฝึกแค่ 40 จุด แต่ validation มี 400 จุด (ไว้ดูของจริง)

In [ ]:
Xbig, ybig = make_moons(n_samples=2000, noise=0.35, random_state=5)
Xs_tr, ys_tr = Xbig[:40], ybig[:40]; Xs_va, ys_va = Xbig[40:440], ybig[40:440]

BIG = [2, 128, 128, 2]
h_over = train(Xs_tr, ys_tr, Xs_va, ys_va, BIG, lr=0.01, epochs=1500, bs=40, opt="adam", seed=1, snapshots=(30, 1500))
best_ep = int(np.argmin(h_over["va_loss"]))
print(f"val loss ต่ำสุดที่ epoch {best_ep} = {h_over['va_loss'][best_ep]:.3f}  | val acc ที่ epoch นั้น = {h_over['va_acc'][best_ep]:.3f}")
print(f"epoch 1500: train loss={h_over['tr_loss'][-1]:.4f} (acc {h_over['tr_acc'][-1]:.2f}) | val loss={h_over['va_loss'][-1]:.3f} (acc {h_over['va_acc'][-1]:.3f})")

fig, axes = plt.subplots(1, 3, figsize=(16, 3.9))
axes[0].plot(h_over["tr_loss"], label="train loss"); axes[0].plot(h_over["va_loss"], label="validation loss"); axes[0].axvline(best_ep, c="gray", ls=":", label=f"best val (epoch {best_ep})")
axes[0].set_ylim(0, max(1.5, min(3, max(h_over["va_loss"][:300])))); axes[0].set_xlabel("epoch"); axes[0].legend(); axes[0].grid(alpha=0.3); axes[0].set_title("OVERFITTING: curves diverge")
plot_boundary(axes[1], h_over["snap"][30], Xs_tr, ys_tr, "epoch 30 (still smooth)"); plot_boundary(axes[2], h_over["snap"][1500], Xs_tr, ys_tr, "epoch 1500 (wiggles to fit noise)")
plt.tight_layout(); plt.show()

### อ่านภาพ overfit
- เส้นฟ้า (train loss) ลงใกล้ 0 → โมเดล "จำ" 40 จุดได้หมด
- เส้นส้ม (validation loss) ลงแล้ว **กลับขึ้น** → โมเดลแย่ลงกับข้อมูลที่ไม่เคยเห็น
- ขอบเขตสุดท้ายวนไปมาล้อมจุดแต่ละจุด (รวมถึง noise) ต่างจากขอบเขตเรียบตอน epoch 30

### วิธีแก้ที่ลองได้เลย: early stopping และ weight decay (L2)
- **Early stopping:** เก็บน้ำหนักตอน validation ดีที่สุด/หยุดเมื่อ val ไม่ดีขึ้นนาน N epoch
- **Weight decay (L2):** เพิ่ม penalty ให้น้ำหนักใหญ่ → ขอบเขตเรียบขึ้น (ในโค้ดคือ `l2=` ที่บวกเข้า gradient `l2·w`)
- **ลดขนาดโมเดล**, **เพิ่มข้อมูล** (ได้ผลที่สุด), **data augmentation**, **dropout** (สุ่มปิด neuron ตอนฝึก — ใช้ใน PyTorch ส่วนที่ 9)

เปรียบเทียบหลายวิธี **บนข้อมูลชุดเดียวกัน** ใช้ validation เป็นผู้ตัดสิน:

In [ ]:
configs = {
    "baseline (big net)":          dict(sizes=BIG, l2=0.0),
    "weight decay l2=0.01":        dict(sizes=BIG, l2=0.01),
    "smaller net [2,8,2]":         dict(sizes=[2, 8, 2], l2=0.0),
}
results = {}
fig, ax = plt.subplots(figsize=(9, 4))
for (name, cfg), c in zip(configs.items(), ["#dc2626", "#16a34a", "#9333ea"]):
    hh = train(Xs_tr, ys_tr, Xs_va, ys_va, cfg["sizes"], lr=0.01, epochs=1500, bs=40, opt="adam", l2=cfg["l2"], seed=1)
    results[name] = hh; ax.plot(hh["va_loss"], c=c, label=name)
ax.set_ylim(0, 2.5); ax.set_xlabel("epoch"); ax.set_ylabel("VALIDATION loss"); ax.legend(); ax.grid(alpha=0.3); ax.set_title("Which fix keeps validation loss low?"); plt.show()

print(f"{'config':26s} {'final val loss':>15s} {'best val loss (epoch)':>24s} {'final val acc':>14s}")
for name, hh in results.items():
    b = int(np.argmin(hh["va_loss"])); print(f"{name:26s} {hh['va_loss'][-1]:15.3f} {hh['va_loss'][b]:>16.3f} ({b:>4d}) {hh['va_acc'][-1]:14.3f}")
print(f"{'early stopping on baseline':26s} {'-':>15s} {h_over['va_loss'][best_ep]:>16.3f} ({best_ep:>4d}) {h_over['va_acc'][best_ep]:14.3f}  <- ใช้น้ำหนักที่ epoch นั้น")

**สังเกต (จากผลที่รันใน seed นี้ ตัวเลขของเธออาจต่างเล็กน้อยตามเวอร์ชันไลบรารี):**
- baseline: val loss ต่ำสุดตั้งแต่ epoch แรกๆ (โมเดลใหญ่เรียนเร็วมาก) แล้ว **ขึ้นไปสูงมาก** ตอนจบ → early stopping ที่ epoch ต้นๆ ให้ผลดีกว่าฝึกจนครบ
- weight decay: ช่วยให้ val loss ปลายทางต่ำกว่า baseline ชัดเจน
- smaller net: best val loss ดีมากแต่ **ยังกลับขึ้นถ้าฝึกนานเกิน** (final แย่กว่า best) → โมเดลเล็กไม่ได้ปลอดภัยจาก overfit ถ้าฝึกนานพอ ควรใช้คู่กับ early stopping
- บทเรียน: วิธีแก้ใช้ **ร่วมกัน** ได้ และ *การทดลองครั้งเดียว (seed เดียว) ไม่ใช่ข้อสรุป* ลองเปลี่ยน `seed` ดูว่าผลนิ่งแค่ไหน

> 🧪 **Mini challenge 8.1:** ปรับ `l2` เป็น 0.001, 0.03, 0.3 หา "จุดที่พอดี" และอธิบายว่า l2 ใหญ่เกินเกิดอะไร (underfit)
>
> 🧪 **Mini challenge 8.2:** เพิ่มข้อมูลฝึกจาก 40 เป็น 400 (`Xbig[:400]`) แล้วรันกราฟ overfit ใหม่ ช่องว่างระหว่าง train/val เปลี่ยนไหม?
>
> 🧪 **Mini challenge 8.3 (เขียนเอง):** เขียนฟังก์ชัน `early_stopping_epoch(va_loss, patience)` ที่คืน epoch ที่ควรหยุด ถ้า val loss ไม่ดีขึ้นต่อเนื่อง `patience` epoch
>
> 📓 **ใน Experiment Log** บันทึก: model, lr, batch, epochs, train/val loss+acc, observation, conclusion (ตามที่โปรเจกต์ของเธอกำหนด)

## 9. PyTorch เทียบกับที่เขียนเอง + TensorFlow/sklearn

> ⚠️ **สถานะการทดสอบ:** สภาพแวดล้อมที่สร้าง notebook นี้ **ติดตั้ง PyTorch ไม่ได้** เซลล์ในส่วนนี้จึง **ตรวจแค่ไวยากรณ์ ไม่ได้รันจริง** ถ้า error ตอนรันบนเครื่องของเธอ (เช่นเวอร์ชัน PyTorch ต่างกัน) บอกได้เลย โค้ดอ้างอิง API มาตรฐานของ PyTorch (`nn.Sequential`, `nn.Linear`, `CrossEntropyLoss`, `optim.Adam`, `DataLoader`) ติดตั้งด้วย `pip install torch` (ดูคำสั่งตามเครื่องที่ pytorch.org/get-started)

**จับคู่สิ่งที่เราเขียนเองกับ PyTorch:**

| ที่เราเขียนเอง (NumPy) | PyTorch |
|---|---|
| `init_params`, `forward` | `nn.Sequential(nn.Linear, nn.ReLU, ...)` |
| softmax + `cross_entropy` | `nn.CrossEntropyLoss()` (รวม softmax ไว้แล้ว — ส่ง logits เข้าไป) |
| `backward(...)` | `loss.backward()` (autograd หา gradient ให้อัตโนมัติ) |
| `opt_step` (Adam) | `torch.optim.Adam(model.parameters(), lr=...)` + `opt.step()` |
| แบ่ง batch ด้วย `perm` | `DataLoader(..., batch_size=32, shuffle=True)` |
| `l2=` | `weight_decay=` ใน optimizer |

In [ ]:
# [needs-torch]  (ตรวจไวยากรณ์เท่านั้น ยังไม่ได้รันในสภาพแวดล้อมสร้าง notebook)
import torch, torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

Xt, yt = torch.tensor(Xtr, dtype=torch.float32), torch.tensor(ytr, dtype=torch.long)
Xv, yv = torch.tensor(Xva, dtype=torch.float32), torch.tensor(yva, dtype=torch.long)
loader = DataLoader(TensorDataset(Xt, yt), batch_size=32, shuffle=True)

model = nn.Sequential(nn.Linear(2, 16), nn.ReLU(), nn.Linear(16, 16), nn.ReLU(), nn.Linear(16, 2))   # logits (ไม่ต้องใส่ softmax)
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)                 # weight_decay=1e-4 ถ้าอยากใช้ L2

tr_hist, va_hist = [], []
for epoch in range(100):
    model.train()
    for xb, yb in loader:
        optimizer.zero_grad()              # ล้าง gradient เก่า (PyTorch สะสมไว้ถ้าไม่ล้าง)
        loss = loss_fn(model(xb), yb)      # forward + loss
        loss.backward()                    # backprop อัตโนมัติ
        optimizer.step()                   # อัปเดตน้ำหนัก
    model.eval()
    with torch.no_grad():                  # ตอนประเมินไม่ต้องคำนวณ gradient
        tr_hist.append(loss_fn(model(Xt), yt).item()); va_hist.append(loss_fn(model(Xv), yv).item())

plt.plot(tr_hist, label="train"); plt.plot(va_hist, label="val"); plt.legend(); plt.title("PyTorch loss curve"); plt.show()

In [ ]:
# [needs-torch]  (ตรวจไวยากรณ์เท่านั้น) — ตัวอย่าง "รูปร่างข้อมูล" ของ LSTM ที่จะใช้ในโปรเจกต์ (ช่วงที่ 6 จะลงรายละเอียด)
import torch, torch.nn as nn
B, T, D, H, n_classes = 8, 30, 63, 64, 20       # 8 คลิป, 30 เฟรม, 63 feature/เฟรม (มือหนึ่งข้าง), ความจำ 64, 20 คำ
x = torch.randn(B, T, D)
lstm = nn.LSTM(input_size=D, hidden_size=H, batch_first=True)
head = nn.Linear(H, n_classes)
out, (h_n, c_n) = lstm(x)
logits = head(h_n[-1])                           # ใช้ความจำสุดท้ายตัดสินว่าเป็นคำไหน
print(out.shape, h_n.shape, logits.shape)        # คาดว่า (8,30,64) (1,8,64) (8,20)

### sklearn vs PyTorch vs TensorFlow (ภาพรวมที่ควรรู้)
| | **scikit-learn** | **PyTorch** | **TensorFlow / Keras** |
|---|---|---|---|
| เหมาะกับ | ML คลาสสิก (ต้นไม้, linear, KNN), เตรียมข้อมูล, metrics | วิจัย + deep learning ยืดหยุ่น (ส่วนใหญ่ในงานวิจัยใช้) | deep learning + deploy (มือถือ/เว็บ) |
| โมเดลลำดับ (LSTM/GRU) | ไม่มี | `nn.LSTM`, `nn.GRU` | `keras.layers.LSTM/GRU` |
| ใช้ GPU | ไม่ | ได้ | ได้ |
| ในโปรเจกต์นี้ | metrics, split, baseline, `GroupKFold` | **โมเดลหลัก (แนะนำ)** | ทางเลือก (เลือกอย่างใดอย่างหนึ่งพอ) |

**คำแนะนำ:** เลือก PyTorch อย่างเดียวสำหรับโปรเจกต์นี้ (ช่วงที่ 5) ใช้ sklearn คู่กันสำหรับ metrics/split ไม่ต้องเรียน TensorFlow ควบ

> ℹ️ ส่วนของ **CNN / backbone / ViT / Ultralytics YOLO-pose** ตามโน้ตของเธอ อยู่ในช่วงที่ 6-7 ของแผน เพราะโปรเจกต์นี้ใช้ landmarks (พิกัด) เป็น input จึงไม่ต้องใช้โมเดลอ่านภาพโดยตรง

## 10. สรุป + Resources + GitHub repos

### สิ่งที่ควรอธิบายได้หลังจบ
1. neuron = `activation(w·x + b)`; layer = หลาย neuron; network = หลาย layer (shape ต้องตรง)
2. loss วัดความผิด (cross-entropy สำหรับ classification), การฝึก = ลด loss
3. gradient = ขยับน้ำหนักแล้ว loss เปลี่ยนเท่าไร; backprop = กฎลูกโซ่ไล่ย้อนหลัง; ตรวจด้วย numerical gradient ได้
4. loop ฝึก: forward → loss → backward → step (ซ้ำทุก batch, ครบข้อมูล = 1 epoch)
5. Adam = momentum + ขนาดก้าวปรับตามพารามิเตอร์; lr คือ hyperparameter สำคัญที่สุด
6. อ่าน loss curve: val กลับขึ้นขณะ train ลง = overfit → early stopping / weight decay / ข้อมูลเพิ่ม / โมเดลเล็กลง

### 📺 วิดีโอ/บทความที่แนะนำ (ค้นยืนยันแล้ว)
| ทำความเข้าใจเรื่อง | แหล่ง |
|---|---|
| Neural network ภาพรวม (3 บทแรกครอบคลุมส่วนที่ 1-4 ของโน้ตนี้) | **3Blue1Brown — Neural Networks**: ตอน *But what is a neural network?*, *Gradient descent*, *What is backpropagation really doing?*, *Backpropagation calculus* — https://3blue1brown.com/neural-networks |
| สร้าง backprop เองตั้งแต่ศูนย์ (ลึกกว่านี้) | **Andrej Karpathy — Neural Networks: Zero to Hero** (วิดีโอ + notebook): https://github.com/karpathy/nn-zero-to-hero — เริ่มที่ตอน *micrograd* |
| Overfit / generalization / loss curve | Google MLCC: https://developers.google.com/machine-learning/crash-course/overfitting/overfitting |
| Optimizer (SGD → Adam) พร้อมโค้ด | Dive into Deep Learning (D2L), บท Adam: https://classic.d2l.ai/chapter_optimization/adam.html |
| ตำราฟรีเน้นโค้ดเดียวกับที่ 3Blue1Brown อ้างถึง | Michael Nielsen — *Neural Networks and Deep Learning* (โค้ดตัวอย่าง): https://github.com/mnielsen/neural-networks-and-deep-learning |
| บล็อกอธิบายภาพ (โครงข่ายประสาท, LSTM) | Chris Olah's blog: http://colah.github.io/ |
| PyTorch เริ่มต้น | https://pytorch.org/tutorials (เริ่มที่ *Learn the Basics*) และโค้ดอยู่ที่ https://github.com/pytorch/tutorials |

### 🗂️ GitHub repos ให้ไปอ่าน/รันต่อ
| repo | ใช้ทำอะไร |
|---|---|
| https://github.com/karpathy/nn-zero-to-hero | คอร์สสร้าง neural network จากศูนย์พร้อม notebook (ผู้เรียนหลายคนทำสรุป/แบบฝึกไว้ใน GitHub ด้วย) |
| https://github.com/jakevdp/PythonDataScienceHandbook | notebook ของ NumPy/Pandas/Matplotlib/scikit-learn ฟรี |
| https://github.com/mnielsen/neural-networks-and-deep-learning | โค้ดจากหนังสือของ Nielsen (โค้ดดั้งเดิมเขียนด้วย Python รุ่นเก่า อาจต้องปรับก่อนรัน — อ่านเพื่อเข้าใจแนวคิด) |
| https://github.com/pytorch/tutorials | บทเรียน PyTorch ทางการ |

### ✅ สิ่งที่อยากให้ทำต่อ (ก่อนไป notebook ช่วงที่ 5-8)
1. ดู 3Blue1Brown 4 ตอนแรก (~1 ชม.) แล้วย้อนกลับมาดู notebook นี้ส่วนที่ 4 อีกรอบ
2. ทำ mini challenge ที่ 5.1, 7.1, 8.1, 8.2 (ทดลองจริงมีค่ากว่าอ่าน)
3. ติดตั้ง PyTorch แล้วรันเซลล์ส่วนที่ 9 บนเครื่องตัวเอง

> ⏭️ **ต่อไป (ตามแผน):** ช่วงที่ 5-7 — PyTorch ให้คล่อง, RNN → LSTM/GRU, MediaPipe landmarks แล้วเข้าสู่ hands-on